Cell 1 — Load Dataset

In [2]:
import json
import pandas as pd

with open("/home/jovyan/Intern Assessment/ZW/RecruitView.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

print("Total Records:", len(df))

Total Records: 2011


Cell 2 — Check Missing Transcripts

In [6]:
report_missing = missing_transcripts[
    ["id", "user_no", "question_id"]
].copy()

report_missing["Issue Type"] = "Missing Transcript"

report_missing = report_missing.rename(columns={
    "id": "Video ID",
    "user_no": "User No",
    "question_id": "Question ID"
})

# Display report table
print(report_missing)

# Summary
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Total Missing Transcripts Found: {len(report_missing)}")

     Video ID User No Question ID          Issue Type
3        0004     277           1  Missing Transcript
383      0384     113          76  Missing Transcript
534      0535      98          16  Missing Transcript
545      0546     124          16  Missing Transcript
721      0722      99          21  Missing Transcript
1003     1004     197          31  Missing Transcript
1175     1176     249           4  Missing Transcript
1335     1336     113          46  Missing Transcript
1376     1377     113          48  Missing Transcript
1409     1410     249           5  Missing Transcript
1445     1446     113          50  Missing Transcript
1656     1657     249           6  Missing Transcript
1966     1967     292          75  Missing Transcript

SUMMARY
Total Missing Transcripts Found: 13


Cell 3 — Check Duplicate Records

**faster view

In [8]:
print("Unique Video IDs:", df["id"].nunique())
print("Total Records:", len(df))

Unique Video IDs: 2011
Total Records: 2011


**check by droping video.mp4 

In [9]:
duplicates = df.drop(columns=["video"]).duplicated(keep=False)

duplicate_report = df[duplicates][
    ["id", "user_no", "question_id"]
].copy()

duplicate_report["Issue Type"] = "Duplicate Record"

print(duplicate_report)

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Total Duplicate Records Found: {len(duplicate_report)}")
print(f"Unique Duplicate Video IDs: {duplicate_report['id'].nunique()}")

Empty DataFrame
Columns: [id, user_no, question_id, Issue Type]
Index: []

SUMMARY
Total Duplicate Records Found: 0
Unique Duplicate Video IDs: 0


**check by video id

In [10]:
duplicate_ids = df[df.duplicated(subset=["id"], keep=False)]

duplicate_report = duplicate_ids[
    ["id", "user_no", "question_id"]
].copy()

duplicate_report["Issue Type"] = "Duplicate Video ID"

print(duplicate_report.sort_values("id"))

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Total Duplicate Records Found: {len(duplicate_report)}")
print(f"Unique Duplicate Video IDs: {duplicate_report['id'].nunique()}")

Empty DataFrame
Columns: [id, user_no, question_id, Issue Type]
Index: []

SUMMARY
Total Duplicate Records Found: 0
Unique Duplicate Video IDs: 0


Cell 4 - Incorrect Filenames

In [11]:
import os
import re

video_folder = "/home/jovyan/Dataset/RecruitView(ZW)/videos"

incorrect_filenames = []

for file in os.listdir(video_folder):
    if file.endswith(".mp4"):
        if not re.match(r"^vid_\d{4}\.mp4$", file):
            incorrect_filenames.append(file)

print("Incorrect Filenames")

for f in incorrect_filenames:
    print(f)

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Total Incorrect Filenames Found: {len(incorrect_filenames)}")

Incorrect Filenames

SUMMARY
Total Incorrect Filenames Found: 0


Cell 5 - Corrupted Videos

In [13]:
import cv2
import os

video_folder = "/home/jovyan/Dataset/RecruitView(ZW)/videos"

corrupted_videos = []

for file in os.listdir(video_folder):
    if file.endswith(".mp4"):

        path = os.path.join(video_folder, file)

        cap = cv2.VideoCapture(path)

        if not cap.isOpened():
            corrupted_videos.append(file)

        cap.release()

print("Corrupted Videos")

for video in corrupted_videos:
    print(video)

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Total Corrupted Videos Found: {len(corrupted_videos)}")

Corrupted Videos

SUMMARY
Total Corrupted Videos Found: 0


Cell 6 - Missing Video Files

In [14]:
import os

missing_videos = []

for _, row in df.iterrows():

    expected_file = f"vid_{row['id']}.mp4"

    full_path = os.path.join(
        "/home/jovyan/Dataset/RecruitView(ZW)/videos",
        expected_file
    )

    if not os.path.exists(full_path):
        missing_videos.append(expected_file)

for video in missing_videos:
    print(video)

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Total Missing Video Files Found: {len(missing_videos)}")


SUMMARY
Total Missing Video Files Found: 0


Cell 7 - Missing Gemini Summaries

In [16]:
missing_summary = df[
    df["gemini_summary"].isna() |
    (df["gemini_summary"].astype(str).str.strip() == "")
]

print(missing_summary[["id","user_no","question_id"]])

print("\nTotal Missing Summaries:", len(missing_summary))

Empty DataFrame
Columns: [id, user_no, question_id]
Index: []

Total Missing Summaries: 0


Cell 8 - Missing Personality Scores

In [17]:
score_columns = [
    "openness",
    "conscientiousness",
    "extraversion",
    "agreeableness",
    "neuroticism",
    "overall_personality",
    "interview_score",
    "answer_score",
    "speaking_skills",
    "confidence_score",
    "facial_expression",
    "overall_performance"
]

missing_scores = df[df[score_columns].isnull().any(axis=1)]

print(missing_scores[["id","user_no"]])

print("\nTotal Records with Missing Scores:", len(missing_scores))

Empty DataFrame
Columns: [id, user_no]
Index: []

Total Records with Missing Scores: 0


Final Report

In [19]:
import pandas as pd

# ====================================================
# ISSUE COUNTS
# ====================================================

total_records = len(df)

# Missing transcripts
missing_transcript_count = len(missing_transcripts)

# Duplicate video IDs (recommended)
duplicate_video_ids = df[df.duplicated(subset=["id"], keep=False)]
duplicate_record_count = duplicate_video_ids["id"].nunique()

# Other checks
incorrect_filename_count = len(incorrect_filenames) if 'incorrect_filenames' in globals() else 0
corrupted_video_count = len(corrupted_videos) if 'corrupted_videos' in globals() else 0
missing_video_count = len(missing_videos) if 'missing_videos' in globals() else 0

# ====================================================
# CLEANING REPORT TABLE
# ====================================================

cleaning_report = pd.DataFrame({
    "Issue Type": [
        "Missing Transcript",
        "Duplicate Records",
        "Incorrect Filenames",
        "Corrupted Videos",
        "Missing Video Files"
    ],
    "Affected Videos": [
        missing_transcript_count,
        duplicate_record_count,
        incorrect_filename_count,
        corrupted_video_count,
        missing_video_count
    ],
    "Description": [
        "Transcript field is empty or missing",
        "Duplicate video IDs detected",
        "Filename does not follow naming convention",
        "Video cannot be opened properly",
        "Metadata exists but video file is missing"
    ],
    "Suggested Data Cleaning Method": [
        "Regenerate transcript using Whisper",
        "Remove duplicate records",
        "Rename files to standard format",
        "Remove or re-download corrupted videos",
        "Restore missing videos or remove metadata"
    ]
})

print("=" * 120)
print("DATA CLEANING REPORT")
print("=" * 120)

print(cleaning_report.to_string(index=False))

# ====================================================
# SUMMARY
# ====================================================

total_issues = (
    missing_transcript_count +
    duplicate_record_count +
    incorrect_filename_count +
    corrupted_video_count +
    missing_video_count
)

print("\n")
print("=" * 120)
print("CLEANING SUMMARY")
print("=" * 120)

print(f"Total Video Records Before Cleaning : {total_records}")
print(f"Total Issues Identified             : {total_issues}")

# Since transcripts can be regenerated,
# we keep all records after cleaning
print(f"Total Video Records After Cleaning  : {total_records}")

print("\nIssue Breakdown")
print("-" * 50)
print(f"Missing Transcripts : {missing_transcript_count}")
print(f"Duplicate Records   : {duplicate_record_count}")
print(f"Incorrect Filenames : {incorrect_filename_count}")
print(f"Corrupted Videos    : {corrupted_video_count}")
print(f"Missing Video Files : {missing_video_count}")

DATA CLEANING REPORT
         Issue Type  Affected Videos                                Description            Suggested Data Cleaning Method
 Missing Transcript               13       Transcript field is empty or missing       Regenerate transcript using Whisper
  Duplicate Records                0               Duplicate video IDs detected                  Remove duplicate records
Incorrect Filenames                0 Filename does not follow naming convention           Rename files to standard format
   Corrupted Videos                0            Video cannot be opened properly    Remove or re-download corrupted videos
Missing Video Files                0  Metadata exists but video file is missing Restore missing videos or remove metadata


CLEANING SUMMARY
Total Video Records Before Cleaning : 2011
Total Issues Identified             : 13
Total Video Records After Cleaning  : 2011

Issue Breakdown
--------------------------------------------------
Missing Transcripts : 13
Duplicat